# Завдання №3

## Підготовка даних

In [1]:
import os
import requests

In [4]:
def load_file(url, file_name):
  file_url=f"{url}{file_name}"
  # Перевірка наявності файлу та завантаження
  if not os.path.exists(file_name):
      print(f"Файл '{file_name}' не знайдено. Починаємо завантаження...")

      try:
          response = requests.get(file_url, stream=True)
          response.raise_for_status()  # Перевірка на помилки HTTP (404, 500 тощо)

          with open(file_name, "wb") as file:
              for chunk in response.iter_content(chunk_size=8192):
                  file.write(chunk)

          print(f"Успішно завантажено та збережено як '{file_name}'.")
      except Exception as e:
          print(f"Помилка під час завантаження файлу: {e}")
  else:
      print(f"Файл '{file_name}' вже існує локально. Завантаження не потрібне.")

In [5]:
url = "https://ai-r.info/data/"

In [6]:
load_file(url, "users.csv")

Файл 'users.csv' не знайдено. Починаємо завантаження...
Успішно завантажено та збережено як 'users.csv'.


In [7]:
load_file(url, "products.csv")

Файл 'products.csv' не знайдено. Починаємо завантаження...
Успішно завантажено та збережено як 'products.csv'.


In [8]:
load_file(url, "purchases.csv")

Файл 'purchases.csv' не знайдено. Починаємо завантаження...
Успішно завантажено та збережено як 'purchases.csv'.


In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("hw03").getOrCreate()

## P1.Завантажте та прочитайте кожен CSV-файл як окремий DataFrame

In [12]:
df_user = spark.read.csv('users.csv', header=True)
df_user.show()

+-------+-------+---+------------------+
|user_id|   name|age|             email|
+-------+-------+---+------------------+
|      1| User_1| 45| user1@example.com|
|      2| User_2| 48| user2@example.com|
|      3| User_3| 36| user3@example.com|
|      4| User_4| 46| user4@example.com|
|      5| User_5| 29| user5@example.com|
|      6| User_6| 39| user6@example.com|
|      7| User_7| 24| user7@example.com|
|      8| User_8| 44| user8@example.com|
|      9| User_9| 27| user9@example.com|
|     10|User_10| 43|user10@example.com|
|     11|User_11| 36|user11@example.com|
|     12|User_12| 24|user12@example.com|
|     13|User_13| 47|user13@example.com|
|     14|User_14| 20|user14@example.com|
|     15|User_15| 30|user15@example.com|
|     16|User_16| 30|user16@example.com|
|     17|User_17| 35|user17@example.com|
|     18|User_18| 45|user18@example.com|
|     19|User_19| 47|user19@example.com|
|     20|User_20| 43|user20@example.com|
+-------+-------+---+------------------+
only showing top

In [13]:
df_product = spark.read.csv('products.csv', header=True)
df_product.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|         1|   Product_1|     Beauty|  8.3|
|         2|   Product_2|       Home|  8.3|
|         3|   Product_3|Electronics|  9.2|
|         4|   Product_4|Electronics|  2.6|
|         5|   Product_5|Electronics|  9.4|
|         6|   Product_6|     Sports|  8.7|
|         7|   Product_7|     Beauty|  8.2|
|         8|   Product_8|     Sports|  1.0|
|         9|   Product_9|     Beauty|  6.0|
|        10|  Product_10|     Sports|  5.4|
|        11|  Product_11|     Sports|  8.6|
|        12|  Product_12|       Home|  1.3|
|        13|  Product_13|     Sports|  7.1|
|        14|  Product_14|   Clothing|  7.4|
|        15|        NULL|       Home|  4.1|
|        16|  Product_16|     Sports|  4.3|
|        17|  Product_17|Electronics|  8.1|
|        18|  Product_18|   Clothing|  4.8|
|        19|  Product_19|   Clothing|  3.5|
|        20|  Product_20|   Clot

In [14]:
df_purchase = spark.read.csv('purchases.csv', header=True)
df_purchase.show()

+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|      date|quantity|
+-----------+-------+----------+----------+--------+
|          1|     52|         9|2022-01-01|       1|
|          2|     93|        37|2022-01-02|       8|
|          3|     15|        33|2022-01-03|       1|
|          4|     72|        42|2022-01-04|       9|
|          5|     61|        44|2022-01-05|       6|
|          6|     21|        24|2022-01-06|       7|
|          7|     83|        15|2022-01-07|       7|
|          8|     87|        32|2022-01-08|       3|
|          9|     75|        32|2022-01-09|       2|
|         10|     75|        24|2022-01-10|       9|
|         11|     88|        41|2022-01-11|       8|
|         12|    100|        49|2022-01-12|       7|
|         13|     24|        49|2022-01-13|       9|
|         14|      3|        12|2022-01-14|       4|
|         15|     22|        39|2022-01-15|       4|
|         16|     53|         2|2022-01-16|   

In [19]:
from pyspark.sql import functions as F
from functools import reduce

def show_null(df):
  # Створюємо умову: (col1 IS NULL) OR (col2 IS NULL) OR ... для всіх колонок
  has_null_condition = reduce(lambda a, b: a | b, [F.col(c).isNull() for c in df.columns])

  # Фільтруємо і рахуємо
  rows_with_nulls = df.filter(has_null_condition).count()
  print(f"Кількість рядків із пропущеними значеннями: {rows_with_nulls}")

  # Якщо потрібно подивитися самі ці рядки:
  df.filter(has_null_condition).show()

## P2.Очистіть дані, видаляючи будь-які рядки з пропущеними значеннями.

In [20]:
show_null(df_user)

Кількість рядків із пропущеними значеннями: 5
+-------+-------+----+------------------+
|user_id|   name| age|             email|
+-------+-------+----+------------------+
|     30|User_30|  30|              NULL|
|     34|   NULL|  48|user34@example.com|
|     81|User_81|NULL|user81@example.com|
|     94|   NULL|  43|user94@example.com|
|     99|User_99|NULL|user99@example.com|
+-------+-------+----+------------------+



In [21]:
show_null(df_product)

Кількість рядків із пропущеними значеннями: 3
+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|        15|        NULL|       Home|  4.1|
|      NULL|  Product_25|Electronics|  7.4|
|        44|  Product_44|       NULL| NULL|
+----------+------------+-----------+-----+



In [22]:
show_null(df_purchase)

Кількість рядків із пропущеними значеннями: 5
+-----------+-------+----------+----------+--------+
|purchase_id|user_id|product_id|      date|quantity|
+-----------+-------+----------+----------+--------+
|         41|   NULL|        38|2022-02-10|       3|
|         50|     39|        33|2022-02-19|    NULL|
|         96|     85|        19|      NULL|       1|
|        150|   NULL|        21|2022-05-30|       4|
|        199|     72|      NULL|2022-07-18|       4|
+-----------+-------+----------+----------+--------+



In [23]:
df_user_cleaned = df_user.dropna()
df_product_cleaned = df_product.dropna()
df_purchase_cleaned = df_purchase.dropna()

## P3.Визначте загальну суму покупок за кожною категорією продуктів.

In [27]:
from pyspark.sql.types import DoubleType, IntegerType

# Примусово приводимо типи до числових, щоб уникнути помилки сумісності
purchases_fixed = df_purchase_cleaned.withColumn("quantity", F.col("quantity").cast(IntegerType()))
products_fixed = df_product_cleaned.withColumn("price", F.col("price").cast(DoubleType()))

# Виконуємо обчислення
result_df = purchases_fixed.join(products_fixed, "product_id") \
    .withColumn("total_amount", F.col("quantity") * F.col("price")) \
    .groupBy("category") \
    .agg(F.sum("total_amount").alias("total_spent")) \
    .withColumn("total_spent", F.round(F.col("total_spent"), 2)) \
    .orderBy(F.desc("total_spent"))

result_df.show()

+-----------+-----------+
|   category|total_spent|
+-----------+-----------+
|     Sports|     1802.5|
|       Home|     1523.5|
|Electronics|     1174.8|
|   Clothing|      790.3|
|     Beauty|      459.9|
+-----------+-----------+



## P4.Визначте суму покупок за кожною категорією продуктів для вікової категорії від 18 до 25 включно.

In [30]:
users_fixed = df_user_cleaned.withColumn("age", F.col("age").cast(IntegerType()))
purchases_fixed = df_purchase_cleaned.withColumn("quantity", F.col("quantity").cast(IntegerType()))
products_fixed = df_product_cleaned.withColumn("price", F.col("price").cast(DoubleType()))

# 1. Фільтруємо користувачів віком від 18 до 25 років включно
# 2. З'єднуємо таблиці: users -> purchases -> products
# 3. Обчислюємо загальну вартість покупки (quantity * price)
# 4. Групуємо за категорією, підсумовуємо та заокруглюємо до 2 знаків
# 5. Сортуємо за спаданням суми
result_df = users_fixed.filter((F.col("age") >= 18) & (F.col("age") <= 25)) \
    .join(purchases_fixed, "user_id") \
    .join(products_fixed, "product_id") \
    .withColumn("total_amount", F.col("quantity") * F.col("price")) \
    .groupBy("category") \
    .agg(F.round(F.sum("total_amount"), 2).alias("total_spent")) \
    .orderBy(F.desc("total_spent"))

result_df.show()

+-----------+-----------+
|   category|total_spent|
+-----------+-----------+
|       Home|      361.1|
|     Sports|      310.5|
|Electronics|      249.6|
|   Clothing|      245.0|
|     Beauty|       41.4|
+-----------+-----------+



## P5.Визначте частку покупок за кожною категорією товарів від сумарних витрат для вікової категорії від 18 до 25 років.

In [35]:
from pyspark.sql.window import Window

users_fixed = df_user_cleaned.withColumn("age", F.col("age").cast(IntegerType()))
purchases_fixed = df_purchase_cleaned.withColumn("quantity", F.col("quantity").cast(IntegerType()))
products_fixed = df_product_cleaned.withColumn("price", F.col("price").cast(DoubleType()))

# 1. Фільтруємо користувачів віком 18-25 років
# 2. Об'єднуємо таблиці
# 3. Рахуємо суму витрат для кожного рядка та групуємо за категорією
category_totals = users_fixed.filter((F.col("age") >= 18) & (F.col("age") <= 25)) \
    .join(purchases_fixed, "user_id") \
    .join(products_fixed, "product_id") \
    .withColumn("total_amount", F.col("quantity") * F.col("price")) \
    .groupBy("category") \
    .agg(F.sum("total_amount").alias("category_spent"))

# Загальна сума витрат для всіх категорій у цій віковій групі
window_all = Window.partitionBy()
result_df = category_totals.withColumn(
        "total_spent_all", F.sum("category_spent").over(window_all)
    ) \
    .withColumn(
        "share_percentage",
        F.round((F.col("category_spent") / F.col("total_spent_all")) * 100, 2)
    ) \
    .withColumn("category_spent", F.round(F.col("category_spent"), 2)) \
    .select("category", "category_spent", "share_percentage") \
    .orderBy(F.desc("share_percentage"))

result_df.show()

+-----------+--------------+----------------+
|   category|category_spent|share_percentage|
+-----------+--------------+----------------+
|       Home|         361.1|            29.9|
|     Sports|         310.5|           25.71|
|Electronics|         249.6|           20.67|
|   Clothing|         245.0|           20.29|
|     Beauty|          41.4|            3.43|
+-----------+--------------+----------------+



## P6.Виберіть 3 категорії продуктів з найвищим відсотком витрат споживачами віком від 18 до 25 років.

In [34]:
users_fixed = df_user_cleaned.withColumn("age", F.col("age").cast(IntegerType()))
purchases_fixed = df_purchase_cleaned.withColumn("quantity", F.col("quantity").cast(IntegerType()))
products_fixed = df_product_cleaned.withColumn("price", F.col("price").cast(DoubleType()))

category_totals = users_fixed.filter((F.col("age") >= 18) & (F.col("age") <= 25)) \
    .join(purchases_fixed, "user_id") \
    .join(products_fixed, "product_id") \
    .withColumn("total_amount", F.col("quantity") * F.col("price")) \
    .groupBy("category") \
    .agg(F.sum("total_amount").alias("category_spent"))

window_all = Window.partitionBy()
top3_df = category_totals.withColumn(
        "total_spent_all", F.sum("category_spent").over(window_all)
    ) \
    .withColumn(
        "share_percentage",
        F.round((F.col("category_spent") / F.col("total_spent_all")) * 100, 2)
    ) \
    .select("category", "share_percentage") \
    .orderBy(F.desc("share_percentage")) \
    .limit(3)

top3_df.show(truncate=False)

+-----------+----------------+
|category   |share_percentage|
+-----------+----------------+
|Home       |29.9            |
|Sports     |25.71           |
|Electronics|20.67           |
+-----------+----------------+



## Висновки

За результатами виконаного аналізу даних за допомогою Apache Spark можна зробити такі головні висновки:

* **Загальні вподобання:** Найбільшу суму коштів усі покупці витрачають на категорії **Sports** (1802.50) та **Home** (1523.5).

* **Молодіжна аудиторія (18–25 років):**

Топ-3 категорії товарів за часткою витрат серед молоді: **Home** (29.90%),
**Sports** (25.71%) та **Electronics** (20.67%).
